# Fertilizer Recommendation System - Exploratory Data Analysis

This notebook explores the agricultural dataset and develops insights for fertilizer recommendations.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Display settings
pd.set_option('display.max_columns', None)
%matplotlib inline

## 1. Load and Inspect Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/raw/fertilizer_data.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

## 2. Exploratory Data Analysis

In [ ]:
# Fertilizer distribution
plt.figure(figsize=(12, 6))
fertilizer_counts = df['Fertilizer'].value_counts()
sns.barplot(x=fertilizer_counts.index, y=fertilizer_counts.values, palette='viridis')
plt.title('Fertilizer Recommendation Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Fertilizer Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nFertilizer Distribution:")
print(fertilizer_counts)

In [ ]:
# Crop distribution
plt.figure(figsize=(10, 6))
crop_counts = df['Crop'].value_counts()
plt.pie(crop_counts.values, labels=crop_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Crop Type Distribution', fontsize=16, fontweight='bold')
plt.axis('equal')
plt.show()

print("\nCrop Distribution:")
print(crop_counts)

In [ ]:
# NPK nutrient distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

nutrients = ['Nitrogen', 'Phosphorus', 'Potassium']
colors = ['#2E86AB', '#A23B72', '#F18F01']

for ax, nutrient, color in zip(axes, nutrients, colors):
    ax.hist(df[nutrient], bins=30, color=color, alpha=0.7, edgecolor='black')
    ax.axvline(df[nutrient].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
    ax.set_xlabel(f'{nutrient} (kg/ha)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{nutrient} Distribution', fontsize=13, fontweight='bold')
    ax.legend()

plt.suptitle('NPK Nutrient Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
numerical_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numerical_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Create enhanced features
df_enhanced = df.copy()

# NPK Ratios
df_enhanced['N_P_ratio'] = df_enhanced['Nitrogen'] / (df_enhanced['Phosphorus'] + 1)
df_enhanced['N_K_ratio'] = df_enhanced['Nitrogen'] / (df_enhanced['Potassium'] + 1)
df_enhanced['P_K_ratio'] = df_enhanced['Phosphorus'] / (df_enhanced['Potassium'] + 1)

# Total NPK
df_enhanced['Total_NPK'] = df_enhanced['Nitrogen'] + df_enhanced['Phosphorus'] + df_enhanced['Potassium']

# Temperature-Humidity Index
df_enhanced['Temp_Humidity_Index'] = df_enhanced['Temperature'] * df_enhanced['Humidity'] / 100

# pH Categories
df_enhanced['pH_acidic'] = (df_enhanced['pH'] < 6.5).astype(int)
df_enhanced['pH_neutral'] = ((df_enhanced['pH'] >= 6.5) & (df_enhanced['pH'] <= 7.5)).astype(int)
df_enhanced['pH_alkaline'] = (df_enhanced['pH'] > 7.5).astype(int)

print(f"Original features: {df.shape[1]}")
print(f"Enhanced features: {df_enhanced.shape[1]}")
print(f"New features added: {df_enhanced.shape[1] - df.shape[1]}")

In [ ]:
# Visualize new features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# N:P Ratio by Fertilizer
fertilizers = df_enhanced['Fertilizer'].unique()[:5]
for fert in fertilizers:
    mask = df_enhanced['Fertilizer'] == fert
    axes[0, 0].scatter(df_enhanced[mask]['N_P_ratio'], 
                      df_enhanced[mask]['Total_NPK'], 
                      label=fert, alpha=0.6)
axes[0, 0].set_xlabel('N:P Ratio')
axes[0, 0].set_ylabel('Total NPK')
axes[0, 0].set_title('N:P Ratio vs Total NPK')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Temperature-Humidity Index
axes[0, 1].hist(df_enhanced['Temp_Humidity_Index'], bins=30, color='skyblue', edgecolor='black')
axes[0, 1].set_xlabel('Temperature-Humidity Index')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Temperature-Humidity Index Distribution')
axes[0, 1].grid(alpha=0.3)

# pH Categories
ph_cats = ['Acidic', 'Neutral', 'Alkaline']
ph_counts = [df_enhanced['pH_acidic'].sum(), 
             df_enhanced['pH_neutral'].sum(), 
             df_enhanced['pH_alkaline'].sum()]
axes[1, 0].bar(ph_cats, ph_counts, color=['red', 'green', 'blue'], alpha=0.7)
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('pH Category Distribution')
axes[1, 0].grid(alpha=0.3)

# NPK Ratios comparison
ratio_data = df_enhanced[['N_P_ratio', 'N_K_ratio', 'P_K_ratio']].mean()
axes[1, 1].bar(ratio_data.index, ratio_data.values, color='orange', alpha=0.7)
axes[1, 1].set_ylabel('Average Ratio')
axes[1, 1].set_title('Average NPK Ratios')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(alpha=0.3)

plt.suptitle('Engineered Features Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Key Insights

From the exploratory analysis, we can observe:

1. **Fertilizer Distribution**: Different fertilizers are recommended based on specific NPK requirements
2. **Crop Patterns**: Each crop type has distinct nutrient requirements
3. **Nutrient Correlations**: Strong relationships exist between certain nutrients
4. **Environmental Factors**: Temperature, humidity, and pH significantly influence recommendations
5. **Feature Engineering**: Created ratios and indices improve model understanding

These insights guide our machine learning model development.

## 5. Next Steps

1. Complete data preprocessing pipeline
2. Train Random Forest classifier
3. Evaluate model performance
4. Deploy prediction system
5. Generate stakeholder reports